# 08 — Feature Engineering

## Objective

This notebook transforms the validated integrated book data into modelling-ready features for subsequent NLP, dimensionality reduction, clustering, and recommendation-system development.

Feature engineering is guided by the findings from exploratory data analysis and statistical analysis rather than by arbitrary transformations.

The objectives are to:

1. preserve the validated canonical book entities and `book_id`
2. create semantic metadata-availability indicators
3. engineer temporal features from publication information
4. transform highly skewed engagement and bibliographic count variables
5. construct interpretable rating and engagement features
6. prepare bibliographic and source-provenance features
7. prepare a consolidated text field for later NLP processing
8. keep original variables alongside engineered variables where appropriate
9. avoid fabrication or statistical imputation of unavailable substantive metadata
10. produce a reproducible feature dataset for subsequent modelling notebooks

### Methodological Principles

Feature engineering follows several principles established in the preceding notebooks:

- missing ratings, descriptions, subjects, engagement values, and other substantive metadata are not fabricated
- extreme observations are not automatically removed merely because they are large
- strongly skewed non-negative count variables may receive `log1p()` transformations while their original values are retained
- ratings and reader interest remain separate because statistical analysis did not establish rating as a reliable proxy for reader interest
- publication recency and reader interest remain separate features
- edition count and reader interest remain separate despite their moderate association
- source-derived subjects, future NLP representations, and future ML clusters are conceptually distinct
- feature engineering must not modify the validated master dataset

The output of this notebook will therefore be a **derived modelling dataset**, while `books_master.csv` remains the authoritative integrated book-level dataset.

In [1]:
# ============================================================
# IMPORTS AND PROJECT PATHS
# ============================================================

from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System"
)

DATA_FINAL = PROJECT_ROOT / "data" / "final"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

books_path = DATA_FINAL / "books_master.csv"
editions_path = DATA_FINAL / "leadershipnow_editions.csv"


print("FILE PATH VALIDATION")
print("=" * 70)

print("books_master.csv:", books_path.exists())
print("leadershipnow_editions.csv:", editions_path.exists())
print("processed directory:", DATA_PROCESSED.exists())

FILE PATH VALIDATION
books_master.csv: True
leadershipnow_editions.csv: True
processed directory: True


In [2]:
# ============================================================
# LOAD VALIDATED DATA
# ============================================================

books = pd.read_csv(books_path)
editions = pd.read_csv(editions_path)


print("DATASET SHAPES")
print("=" * 70)

print(f"Canonical books: {books.shape}")
print(f"Edition records: {editions.shape}")


# ------------------------------------------------------------
# Integrity validation
# ------------------------------------------------------------

assert len(books) == 2067
assert books["book_id"].nunique() == 2067
assert books["book_id"].isna().sum() == 0

assert len(editions) == 1120
assert editions["book_id"].nunique() == 1120


print("\n✓ SOURCE DATASETS VALIDATED")

DATASET SHAPES
Canonical books: (2067, 18)
Edition records: (1120, 23)

✓ SOURCE DATASETS VALIDATED


## 1. Feature Dataset Initialization

Feature engineering is performed on a copy of the validated master dataset.

The original `books_master.csv` file is not modified. This preserves reproducibility and maintains a clear distinction between:

- integrated source data
- engineered modelling features
- later NLP representations
- later machine-learning outputs

In [3]:
# ============================================================
# INITIALIZE FEATURE DATASET
# ============================================================

features = books.copy()


print("FEATURE DATASET INITIALIZATION")
print("=" * 70)

print(f"Rows:              {len(features):,}")
print(f"Columns:           {features.shape[1]}")
print(f"Unique book IDs:   {features['book_id'].nunique():,}")
print(f"Missing book IDs:  {features['book_id'].isna().sum():,}")
print(f"Duplicate IDs:     {features['book_id'].duplicated().sum():,}")


assert len(features) == 2067
assert features["book_id"].nunique() == 2067
assert features["book_id"].duplicated().sum() == 0


print("\n✓ FEATURE DATASET INITIALIZED")

FEATURE DATASET INITIALIZATION
Rows:              2,067
Columns:           18
Unique book IDs:   2,067
Missing book IDs:  0
Duplicate IDs:     0

✓ FEATURE DATASET INITIALIZED


In [4]:
# ============================================================
# SEMANTIC AVAILABILITY FUNCTIONS
# ============================================================

MISSING_PLACEHOLDERS = {
    "",
    "[]",
    "{}",
    "nan",
    "none",
    "null",
    "na",
    "n/a"
}


def semantic_available(value):
    """
    Return True when a value contains meaningful metadata,
    rather than a null or serialized empty placeholder.
    """

    if pd.isna(value):
        return False

    text = str(value).strip().lower()

    return text not in MISSING_PLACEHOLDERS


def parse_list_field(value):
    """
    Convert serialized list fields into Python lists while
    preserving legitimate text values.
    """

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    text = str(value).strip()

    if text.lower() in MISSING_PLACEHOLDERS:
        return []

    try:

        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):

            return [
                str(item).strip()
                for item in parsed
                if str(item).strip()
            ]

    except (ValueError, SyntaxError):
        pass

    return [text]


print("✓ SEMANTIC AVAILABILITY FUNCTIONS CREATED")

✓ SEMANTIC AVAILABILITY FUNCTIONS CREATED


In [5]:
# ============================================================
# METADATA AVAILABILITY FEATURES
# ============================================================

features["has_authors"] = (
    features["authors"]
    .apply(semantic_available)
    .astype(int)
)

features["has_description"] = (
    features["description"]
    .apply(semantic_available)
    .astype(int)
)

features["has_subjects"] = (
    features["subjects"]
    .apply(semantic_available)
    .astype(int)
)

features["has_cover"] = (
    features["cover_url"]
    .apply(semantic_available)
    .astype(int)
)

features["has_rating"] = (
    features["average_rating"]
    .notna()
    .astype(int)
)

features["has_engagement"] = (
    features[
        [
            "want_to_read_count",
            "currently_reading_count",
            "already_read_count"
        ]
    ]
    .notna()
    .any(axis=1)
    .astype(int)
)


# At least one substantive text field beyond title
features["has_extended_text"] = (
    (
        features["has_description"] == 1
    )
    |
    (
        features["has_subjects"] == 1
    )
).astype(int)


availability_summary = pd.DataFrame({
    "feature": [
        "has_authors",
        "has_description",
        "has_subjects",
        "has_cover",
        "has_rating",
        "has_engagement",
        "has_extended_text"
    ]
})


availability_summary["available_n"] = (
    availability_summary["feature"]
    .map(features.sum())
)

availability_summary["coverage_pct"] = (
    availability_summary["available_n"]
    / len(features)
    * 100
)


display(
    availability_summary.style.format({
        "coverage_pct": "{:.2f}%"
    })
)

,feature,available_n,coverage_pct
0,has_authors,2059,99.61%
1,has_description,192,9.29%
2,has_subjects,826,39.96%
3,has_cover,1888,91.34%
4,has_rating,282,13.64%
5,has_engagement,807,39.04%
6,has_extended_text,829,40.11%


## 2. Temporal Feature Engineering

Two publication-year variables are available, but they represent different bibliographic concepts:

- `first_publish_year` represents the earliest known publication year associated with a work in Open Library.
- `publication_year_observed` represents the publication year observed in the LeadershipNow edition data.

These variables are therefore retained separately rather than merged into a single publication-year field.

Using the project's fixed reference year of **2026**, the following derived variables are created:

- `book_age_from_first_publish` — years since the earliest known publication
- `years_since_observed_publication` — years since the observed edition publication
- availability indicators for each temporal measure

The fixed reference year improves reproducibility by preventing the engineered values from changing automatically when the notebook is rerun in a future calendar year.

Missing publication years remain missing and are not statistically imputed.

In [6]:
# ============================================================
# TEMPORAL FEATURE ENGINEERING
# ============================================================

REFERENCE_YEAR = 2026


# ------------------------------------------------------------
# Ensure publication-year fields are numeric
# ------------------------------------------------------------

features["first_publish_year"] = pd.to_numeric(
    features["first_publish_year"],
    errors="coerce"
)

features["publication_year_observed"] = pd.to_numeric(
    features["publication_year_observed"],
    errors="coerce"
)


# ------------------------------------------------------------
# Availability indicators
# ------------------------------------------------------------

features["has_first_publish_year"] = (
    features["first_publish_year"]
    .notna()
    .astype(int)
)

features["has_observed_publication_year"] = (
    features["publication_year_observed"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# Derived age / recency features
# ------------------------------------------------------------

features["book_age_from_first_publish"] = (
    REFERENCE_YEAR - features["first_publish_year"]
)

features["years_since_observed_publication"] = (
    REFERENCE_YEAR - features["publication_year_observed"]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    features.loc[
        features["first_publish_year"].notna(),
        "book_age_from_first_publish"
    ] >= 0
).all()

assert (
    features.loc[
        features["publication_year_observed"].notna(),
        "years_since_observed_publication"
    ] >= 0
).all()


temporal_summary = pd.DataFrame({
    "feature": [
        "first_publish_year",
        "book_age_from_first_publish",
        "publication_year_observed",
        "years_since_observed_publication"
    ],
    "n": [
        features["first_publish_year"].notna().sum(),
        features["book_age_from_first_publish"].notna().sum(),
        features["publication_year_observed"].notna().sum(),
        features["years_since_observed_publication"].notna().sum()
    ],
    "min": [
        features["first_publish_year"].min(),
        features["book_age_from_first_publish"].min(),
        features["publication_year_observed"].min(),
        features["years_since_observed_publication"].min()
    ],
    "median": [
        features["first_publish_year"].median(),
        features["book_age_from_first_publish"].median(),
        features["publication_year_observed"].median(),
        features["years_since_observed_publication"].median()
    ],
    "max": [
        features["first_publish_year"].max(),
        features["book_age_from_first_publish"].max(),
        features["publication_year_observed"].max(),
        features["years_since_observed_publication"].max()
    ]
})


print("TEMPORAL FEATURE ENGINEERING")
print("=" * 75)
print(f"Reference year: {REFERENCE_YEAR}")
print()

display(
    temporal_summary.style.format({
        "min": "{:.0f}",
        "median": "{:.0f}",
        "max": "{:.0f}"
    })
)

TEMPORAL FEATURE ENGINEERING
Reference year: 2026



,feature,n,min,median,max
0,first_publish_year,947,1900,2004,2025
1,book_age_from_first_publish,947,1,22,126
2,publication_year_observed,1117,2022,2023,2025
3,years_since_observed_publication,1117,1,3,4


## 3. Transformation of Skewed Numerical Features

Exploratory analysis identified substantial right-skew in several count-based variables, particularly reader-engagement measures and edition count.

The statistical analysis also demonstrated that relationships involving these variables can differ between raw and transformed scales.

Rather than deleting large observations as outliers, logarithmic transformations are created using `log1p(x)`, which is defined for zero-valued counts.

The original variables are retained alongside their transformed versions.

Variables transformed in this stage are:

- `ratings_count`
- `want_to_read_count`
- `currently_reading_count`
- `already_read_count`
- `edition_count`

`average_rating` is not log-transformed because it is a bounded rating measure rather than a count variable.

Missing values remain missing and are not imputed.

In [7]:
# ============================================================
# LOG TRANSFORMATION OF SKEWED COUNT FEATURES
# ============================================================

count_features = [
    "ratings_count",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "edition_count"
]


# ------------------------------------------------------------
# Convert to numeric and validate non-negative values
# ------------------------------------------------------------

for col in count_features:

    features[col] = pd.to_numeric(
        features[col],
        errors="coerce"
    )

    observed = features[col].dropna()

    assert (observed >= 0).all(), (
        f"{col} contains negative values."
    )


# ------------------------------------------------------------
# Create log1p features
# ------------------------------------------------------------

for col in count_features:

    features[f"log1p_{col}"] = np.log1p(
        features[col]
    )


# ------------------------------------------------------------
# Compare skewness before and after transformation
# ------------------------------------------------------------

transformation_summary = []

for col in count_features:

    log_col = f"log1p_{col}"

    transformation_summary.append({
        "feature": col,
        "n": features[col].notna().sum(),
        "raw_min": features[col].min(),
        "raw_median": features[col].median(),
        "raw_max": features[col].max(),
        "raw_skew": features[col].skew(),
        "log1p_skew": features[log_col].skew()
    })


transformation_summary = pd.DataFrame(
    transformation_summary
)


print("COUNT FEATURE TRANSFORMATION")
print("=" * 85)

display(
    transformation_summary.style.format({
        "raw_min": "{:.2f}",
        "raw_median": "{:.2f}",
        "raw_max": "{:.2f}",
        "raw_skew": "{:.3f}",
        "log1p_skew": "{:.3f}"
    })
)

COUNT FEATURE TRANSFORMATION


,feature,n,raw_min,raw_median,raw_max,raw_skew,log1p_skew
0,ratings_count,282,0.00,1.50,278.00,13.485,1.721
1,want_to_read_count,807,0.00,9.00,6754.00,20.845,0.641
2,currently_reading_count,807,0.00,1.00,643.00,21.878,1.525
3,already_read_count,807,0.00,0.00,328.00,17.867,2.154
4,edition_count,950,1.00,4.00,95.00,3.956,0.616


## 4. Reader-Engagement Feature Engineering

Open Library provides three reader-engagement indicators:

- `want_to_read_count` — prospective reader interest
- `currently_reading_count` — active reading engagement
- `already_read_count` — previous or completed reading engagement

These measures represent different behavioural states and are therefore retained individually.

An additional `total_reader_engagement` feature is created as the sum of the three observed engagement counts. This provides an aggregate measure of recorded reader activity without replacing the original dimensions.

A logarithmically transformed version is also created because aggregate engagement is expected to remain strongly right-skewed.

Missing engagement metadata is **not interpreted as zero engagement**. Aggregate engagement is calculated only for books for which engagement data are available.

In [8]:
# ============================================================
# READER-ENGAGEMENT FEATURES
# ============================================================

engagement_cols = [
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count"
]


# ------------------------------------------------------------
# Total recorded engagement
# ------------------------------------------------------------

features["total_reader_engagement"] = (
    features[engagement_cols]
    .sum(axis=1, min_count=1)
)


# ------------------------------------------------------------
# Log-transformed aggregate engagement
# ------------------------------------------------------------

features["log1p_total_reader_engagement"] = np.log1p(
    features["total_reader_engagement"]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    features["total_reader_engagement"].notna().sum()
    ==
    features["has_engagement"].sum()
)

assert (
    features.loc[
        features["has_engagement"] == 0,
        "total_reader_engagement"
    ].isna().all()
)


engagement_summary = pd.DataFrame({
    "feature": [
        "want_to_read_count",
        "currently_reading_count",
        "already_read_count",
        "total_reader_engagement",
        "log1p_total_reader_engagement"
    ],
    "n": [
        features["want_to_read_count"].notna().sum(),
        features["currently_reading_count"].notna().sum(),
        features["already_read_count"].notna().sum(),
        features["total_reader_engagement"].notna().sum(),
        features["log1p_total_reader_engagement"].notna().sum()
    ],
    "mean": [
        features["want_to_read_count"].mean(),
        features["currently_reading_count"].mean(),
        features["already_read_count"].mean(),
        features["total_reader_engagement"].mean(),
        features["log1p_total_reader_engagement"].mean()
    ],
    "median": [
        features["want_to_read_count"].median(),
        features["currently_reading_count"].median(),
        features["already_read_count"].median(),
        features["total_reader_engagement"].median(),
        features["log1p_total_reader_engagement"].median()
    ],
    "max": [
        features["want_to_read_count"].max(),
        features["currently_reading_count"].max(),
        features["already_read_count"].max(),
        features["total_reader_engagement"].max(),
        features["log1p_total_reader_engagement"].max()
    ],
    "skew": [
        features["want_to_read_count"].skew(),
        features["currently_reading_count"].skew(),
        features["already_read_count"].skew(),
        features["total_reader_engagement"].skew(),
        features["log1p_total_reader_engagement"].skew()
    ]
})


print("READER-ENGAGEMENT FEATURE SUMMARY")
print("=" * 85)

display(
    engagement_summary.style.format({
        "mean": "{:.2f}",
        "median": "{:.2f}",
        "max": "{:.2f}",
        "skew": "{:.3f}"
    })
)

READER-ENGAGEMENT FEATURE SUMMARY


,feature,n,mean,median,max,skew
0,want_to_read_count,807,47.49,9.00,6754.00,20.845
1,currently_reading_count,807,3.93,1.00,643.00,21.878
2,already_read_count,807,2.31,0.00,328.00,17.867
3,total_reader_engagement,807,53.73,11.00,7725.00,20.882
4,log1p_total_reader_engagement,807,2.59,2.48,8.95,0.634


## 5. Bibliographic and Edition Feature Engineering

The canonical feature dataset contains one row per book, while the LeadershipNow edition dataset contains edition-level bibliographic information.

To preserve the canonical **book-level grain**, edition information is summarized by `book_id` before being merged into the feature dataset.

The engineered bibliographic features include:

- observed page count
- logarithmically transformed page count
- hardcover indicator
- paperback indicator
- edition-metadata availability

The original format label is also retained for interpretation.

No missing page count or edition format is imputed. Books without LeadershipNow edition metadata remain missing for these variables.

This is particularly important because edition metadata availability is source-dependent rather than random across the complete catalogue.

In [12]:
# ============================================================
# EDITION-GRAIN VALIDATION
# ============================================================

print("EDITION DATA VALIDATION")
print("=" * 75)

print(f"Edition rows:             {len(editions):,}")
print(f"Unique book IDs:          {editions['book_id'].nunique():,}")
print(f"Duplicate book IDs:       {editions['book_id'].duplicated().sum():,}")

print("\nFORMAT DISTRIBUTION")
print("=" * 75)

print(
    editions["format"]
    .fillna("Missing")
    .value_counts(dropna=False)
)

print("\nPAGE COUNT COVERAGE")
print("=" * 75)

print(f"Available: {editions['page_count'].notna().sum():,}")
print(f"Missing:   {editions['page_count'].isna().sum():,}")

print("\nRELEVANT COLUMN VALIDATION")
print("=" * 75)

required_edition_columns = [
    "book_id",
    "format",
    "page_count"
]

for col in required_edition_columns:
    print(f"{col:<15}: {col in editions.columns}")

assert all(
    col in editions.columns
    for col in required_edition_columns
)

print("\n✓ REQUIRED EDITION FEATURES AVAILABLE")

EDITION DATA VALIDATION
Edition rows:             1,120
Unique book IDs:          1,120
Duplicate book IDs:       0

FORMAT DISTRIBUTION
format
Hardcover    924
Paperback    196
Name: count, dtype: int64

PAGE COUNT COVERAGE
Available: 1,119
Missing:   1

RELEVANT COLUMN VALIDATION
book_id        : True
format         : True
page_count     : True

✓ REQUIRED EDITION FEATURES AVAILABLE


In [13]:
# ============================================================
# BOOK-LEVEL EDITION FEATURES
# ============================================================

edition_features = (
    editions[
        [
            "book_id",
            "format",
            "page_count"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Numeric page count
# ------------------------------------------------------------

edition_features["page_count"] = pd.to_numeric(
    edition_features["page_count"],
    errors="coerce"
)


# ------------------------------------------------------------
# Metadata availability
# ------------------------------------------------------------

edition_features["has_edition_metadata"] = 1

edition_features["has_page_count"] = (
    edition_features["page_count"]
    .notna()
    .astype(int)
)


# ------------------------------------------------------------
# Format indicators
# ------------------------------------------------------------

format_normalized = (
    edition_features["format"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

edition_features["is_hardcover"] = (
    format_normalized == "hardcover"
).astype(int)

edition_features["is_paperback"] = (
    format_normalized == "paperback"
).astype(int)


# ------------------------------------------------------------
# Page-count transformation
# ------------------------------------------------------------

assert (
    edition_features["page_count"]
    .dropna()
    .ge(0)
    .all()
)

edition_features["log1p_page_count"] = np.log1p(
    edition_features["page_count"]
)


print("BOOK-LEVEL EDITION FEATURE TABLE")
print("=" * 75)

print(f"Rows:             {len(edition_features):,}")
print(f"Unique book IDs:  {edition_features['book_id'].nunique():,}")

display(edition_features.head())

BOOK-LEVEL EDITION FEATURE TABLE
Rows:             1,120
Unique book IDs:  1,120


,book_id,format,page_count,has_edition_metadata,has_page_count,is_hardcover,is_paperback,log1p_page_count
0,BOOK00951,Hardcover,160.000,1,1,1,0,5.081
1,BOOK00952,Hardcover,848.000,1,1,1,0,6.744
2,BOOK00953,Hardcover,320.000,1,1,1,0,5.771
3,BOOK00954,Hardcover,672.000,1,1,1,0,6.512
4,BOOK00955,Hardcover,288.000,1,1,1,0,5.666


In [14]:
# ============================================================
# MERGE EDITION FEATURES ONTO CANONICAL BOOKS
# ============================================================

rows_before = len(features)

features = features.merge(
    edition_features,
    on="book_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_edition")
)

rows_after = len(features)


# ------------------------------------------------------------
# Fill only structural indicator variables
# ------------------------------------------------------------

indicator_cols = [
    "has_edition_metadata",
    "has_page_count",
    "is_hardcover",
    "is_paperback"
]

features[indicator_cols] = (
    features[indicator_cols]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# Integrity validation
# ------------------------------------------------------------

assert rows_before == 2067
assert rows_after == 2067
assert features["book_id"].nunique() == 2067
assert features["book_id"].duplicated().sum() == 0

assert features["has_edition_metadata"].sum() == 1120
assert features["has_page_count"].sum() == 1119


print("EDITION FEATURE MERGE VALIDATION")
print("=" * 75)

print(f"Rows before merge:        {rows_before:,}")
print(f"Rows after merge:         {rows_after:,}")
print(f"Unique books:             {features['book_id'].nunique():,}")
print(f"Books with edition data:  {features['has_edition_metadata'].sum():,}")
print(f"Books with page count:    {features['has_page_count'].sum():,}")
print(f"Hardcover books:          {features['is_hardcover'].sum():,}")
print(f"Paperback books:          {features['is_paperback'].sum():,}")

print("\n✓ CANONICAL BOOK GRAIN PRESERVED")

EDITION FEATURE MERGE VALIDATION
Rows before merge:        2,067
Rows after merge:         2,067
Unique books:             2,067
Books with edition data:  1,120
Books with page count:    1,119
Hardcover books:          924
Paperback books:          196

✓ CANONICAL BOOK GRAIN PRESERVED


## 6. Categorical and Textual Feature Audit

Categorical metadata are evaluated before encoding.

High-cardinality variables such as author, publisher, and subject should not automatically be converted into conventional one-hot encoded columns. Doing so could create a very large and sparse feature matrix while assigning equal categorical importance to rare and common values.

This stage therefore examines:

- author multiplicity and cardinality
- subject multiplicity and cardinality
- publisher cardinality
- edition-format cardinality

Authors and subjects are also important semantic information and may ultimately be more appropriate for NLP/vector-based representation than conventional categorical encoding.

No categories are removed or grouped at this stage.

In [15]:
# ============================================================
# CATEGORICAL CARDINALITY AUDIT
# ============================================================

# Parse serialized list fields
author_lists = features["authors"].apply(parse_list_field)
subject_lists = features["subjects"].apply(parse_list_field)


# ------------------------------------------------------------
# Flatten author and subject values
# ------------------------------------------------------------

all_authors = [
    author
    for authors in author_lists
    for author in authors
]

all_subjects = [
    subject
    for subjects in subject_lists
    for subject in subjects
]


# ------------------------------------------------------------
# Publisher values from observed editions
# ------------------------------------------------------------

publisher_values = (
    editions["publisher"]
    .dropna()
    .astype(str)
    .str.strip()
)

publisher_values = publisher_values[
    publisher_values != ""
]


# ------------------------------------------------------------
# Cardinality summary
# ------------------------------------------------------------

cardinality_summary = pd.DataFrame({
    "feature": [
        "authors",
        "subjects",
        "publisher",
        "format"
    ],

    "records_with_values": [
        author_lists.apply(len).gt(0).sum(),
        subject_lists.apply(len).gt(0).sum(),
        publisher_values.index.nunique(),
        editions["format"].notna().sum()
    ],

    "unique_values": [
        pd.Series(all_authors).nunique(),
        pd.Series(all_subjects).nunique(),
        publisher_values.nunique(),
        editions["format"].dropna().nunique()
    ]
})


cardinality_summary["coverage_pct"] = [
    author_lists.apply(len).gt(0).mean() * 100,
    subject_lists.apply(len).gt(0).mean() * 100,
    len(publisher_values) / len(editions) * 100,
    editions["format"].notna().mean() * 100
]


print("CATEGORICAL CARDINALITY AUDIT")
print("=" * 80)

display(
    cardinality_summary.style.format({
        "coverage_pct": "{:.2f}%"
    })
)


print("\nMOST FREQUENT AUTHORS")
print("=" * 80)

display(
    pd.Series(all_authors)
    .value_counts()
    .head(10)
    .rename_axis("author")
    .reset_index(name="book_mentions")
)


print("\nMOST FREQUENT SUBJECTS")
print("=" * 80)

display(
    pd.Series(all_subjects)
    .value_counts()
    .head(15)
    .rename_axis("subject")
    .reset_index(name="book_mentions")
)


print("\nMOST FREQUENT PUBLISHERS")
print("=" * 80)

display(
    publisher_values
    .value_counts()
    .head(15)
    .rename_axis("publisher")
    .reset_index(name="edition_mentions")
)

CATEGORICAL CARDINALITY AUDIT


,feature,records_with_values,unique_values,coverage_pct
0,authors,2059,2289,99.61%
1,subjects,826,2300,39.96%
2,publisher,1120,302,100.00%
3,format,1120,2,100.00%



MOST FREQUENT AUTHORS


,author,book_mentions
0,Michael Armstrong,12
1,Gary Dessler,8
2,Jay Heizer,7
3,John C. Maxwell,6
4,Bernard M. Bass,6
5,Ronald B. Adler,6
6,Daniel Goleman,6
7,Bruce J. Avolio,5
8,Fred R. David,5
9,Stephen P. Robbins,5



MOST FREQUENT SUBJECTS


,subject,book_mentions
0,Management,247
1,Leadership,186
2,Personnel management,95
3,Strategic planning,86
4,Gestion,75
5,BUSINESS & ECONOMICS,67
6,Industrial management,64
7,Organizational behavior,64
8,Organizational change,61
9,Case studies,57



MOST FREQUENT PUBLISHERS


,publisher,edition_mentions
0,Wiley,126
1,Harvard Business Review Press,67
2,Berrett-Koehler Publishers,41
3,Matt Holt,33
4,Portfolio,28
5,Harper Business,26
6,HarperCollins Leadership,22
7,St. Martin's Press,19
8,Amplify Publishing,19
9,ForbesBooks,19


## 7. Feature Role Classification

The cardinality audit demonstrates that the available variables should not all be treated using the same modelling strategy.

### A. Content / NLP Features

The following variables contain semantic information and will be prepared for text-based representation:

- title
- authors
- subjects
- description

Authors contain 2,289 distinct raw values and subjects contain 2,300 distinct labels. Conventional one-hot encoding would therefore create high-dimensional sparse representations while failing to capture semantic relationships between terms.

These fields will instead contribute to the content representation developed in the NLP stage.

### B. Numerical Modelling Candidates

The following engineered variables are potential inputs to later numerical analysis, PCA, or clustering where sufficient coverage is available:

- average rating
- log-transformed rating count
- log-transformed reader-engagement measures
- log-transformed total reader engagement
- log-transformed edition count
- book age
- observed-publication recency
- log-transformed page count
- edition-format indicators

These variables will not automatically be placed into a single model. Coverage, missingness, redundancy, scaling, and modelling purpose will be evaluated before constructing each modelling matrix.

### C. Categorical / Filter Metadata

Publisher contains 302 distinct values across the observed LeadershipNow editions. It is retained as bibliographic metadata rather than expanded into hundreds of dummy variables at this stage.

Edition format contains only two observed categories and has already been represented using binary indicators.

### D. Display and Provenance Metadata

Identifiers, cover URLs, Open Library keys, source indicators, and similar provenance fields remain useful for application display, traceability, filtering, and validation but are not treated as semantic or numerical predictors merely because they are available.

This separation prevents identifiers and high-cardinality metadata from entering machine-learning models without a defensible analytical purpose.

In [16]:
# ============================================================
# TEXT FEATURE PREPARATION
# ============================================================

def list_to_text(value):
    """
    Convert a serialized list field into a space-separated
    text representation.
    """
    values = parse_list_field(value)

    return " ".join(
        str(item).strip()
        for item in values
        if str(item).strip()
    )


def clean_text_component(value):
    """
    Apply minimal structural text cleaning without performing
    NLP-specific stemming, lemmatization, or stop-word removal.
    """

    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in MISSING_PLACEHOLDERS:
        return ""

    # Normalize whitespace only
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ------------------------------------------------------------
# Prepare individual content components
# ------------------------------------------------------------

features["title_text"] = (
    features["canonical_title"]
    .apply(clean_text_component)
)

features["authors_text"] = (
    features["authors"]
    .apply(list_to_text)
    .apply(clean_text_component)
)

features["subjects_text"] = (
    features["subjects"]
    .apply(list_to_text)
    .apply(clean_text_component)
)

features["description_text"] = (
    features["description"]
    .apply(clean_text_component)
)


# ------------------------------------------------------------
# Combined content document
# ------------------------------------------------------------

features["content_text"] = (
    features[
        [
            "title_text",
            "authors_text",
            "subjects_text",
            "description_text"
        ]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# ------------------------------------------------------------
# Basic text-size features
# ------------------------------------------------------------

features["content_word_count"] = (
    features["content_text"]
    .str.split()
    .str.len()
)

features["content_char_count"] = (
    features["content_text"]
    .str.len()
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert features["title_text"].ne("").sum() == 2067
assert features["content_text"].ne("").sum() == 2067
assert features["content_word_count"].gt(0).all()


print("TEXT FEATURE PREPARATION")
print("=" * 80)

print(
    f"Books with title text:       "
    f"{features['title_text'].ne('').sum():,}"
)

print(
    f"Books with author text:      "
    f"{features['authors_text'].ne('').sum():,}"
)

print(
    f"Books with subject text:     "
    f"{features['subjects_text'].ne('').sum():,}"
)

print(
    f"Books with description text: "
    f"{features['description_text'].ne('').sum():,}"
)

print(
    f"Books with combined content: "
    f"{features['content_text'].ne('').sum():,}"
)

print("\nCONTENT LENGTH")
print("=" * 80)

print(
    features[
        [
            "content_word_count",
            "content_char_count"
        ]
    ]
    .describe()
    .round(2)
)

TEXT FEATURE PREPARATION
Books with title text:       2,067
Books with author text:      2,059
Books with subject text:     826
Books with description text: 192
Books with combined content: 2,067

CONTENT LENGTH
       content_word_count  content_char_count
count           2,067.000           2,067.000
mean               22.790             170.580
std                48.980             349.690
min                 3.000              14.000
25%                 5.000              35.000
50%                 8.000              53.000
75%                15.000             129.500
max               533.000           3,635.000


## 8. Content Richness and Metadata Depth

Although every canonical book contains sufficient information to construct a non-empty `content_text` field, textual metadata coverage is not uniform.

A book represented only by its title and author contains substantially less semantic information than a book represented by title, authors, subjects, and description.

To make this difference explicit, a `content_component_count` variable is created from four textual components:

1. title
2. authors
3. subjects
4. description

The resulting count ranges from 1 to 4.

A descriptive `content_depth` label is also created:

- **Minimal** — 1 component
- **Basic** — 2 components
- **Enriched** — 3 components
- **Rich** — 4 components

These labels describe **metadata completeness only**. They do not measure book quality, relevance, or recommendation value.

Word and character counts are retained as additional measures of document size.

This information can later be used to evaluate whether recommendation performance differs between metadata-rich and metadata-sparse books.

In [17]:
# ============================================================
# CONTENT RICHNESS FEATURES
# ============================================================

features["has_title_text"] = (
    features["title_text"]
    .ne("")
    .astype(int)
)

features["has_author_text"] = (
    features["authors_text"]
    .ne("")
    .astype(int)
)

features["has_subject_text"] = (
    features["subjects_text"]
    .ne("")
    .astype(int)
)

features["has_description_text"] = (
    features["description_text"]
    .ne("")
    .astype(int)
)


# ------------------------------------------------------------
# Number of available textual components
# ------------------------------------------------------------

text_component_cols = [
    "has_title_text",
    "has_author_text",
    "has_subject_text",
    "has_description_text"
]

features["content_component_count"] = (
    features[text_component_cols]
    .sum(axis=1)
)


# ------------------------------------------------------------
# Descriptive metadata-depth label
# ------------------------------------------------------------

depth_map = {
    1: "Minimal",
    2: "Basic",
    3: "Enriched",
    4: "Rich"
}

features["content_depth"] = (
    features["content_component_count"]
    .map(depth_map)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert features["content_component_count"].between(1, 4).all()
assert features["content_depth"].notna().all()


content_depth_summary = (
    features["content_depth"]
    .value_counts()
    .reindex(
        ["Minimal", "Basic", "Enriched", "Rich"],
        fill_value=0
    )
    .rename_axis("content_depth")
    .reset_index(name="books")
)

content_depth_summary["coverage_pct"] = (
    content_depth_summary["books"]
    / len(features)
    * 100
)


print("CONTENT RICHNESS SUMMARY")
print("=" * 75)

display(
    content_depth_summary.style.format({
        "coverage_pct": "{:.2f}%"
    })
)


print("\nCOMPONENT COUNT DISTRIBUTION")
print("=" * 75)

display(
    features["content_component_count"]
    .value_counts()
    .sort_index()
    .rename_axis("components_available")
    .reset_index(name="books")
)

CONTENT RICHNESS SUMMARY


,content_depth,books,coverage_pct
0,Minimal,3,0.15%
1,Basic,1240,59.99%
2,Enriched,635,30.72%
3,Rich,189,9.14%



COMPONENT COUNT DISTRIBUTION


,components_available,books
0,1,3
1,2,1240
2,3,635
3,4,189


## 9. Content Richness by Data Source

The integrated catalogue combines two sources with different metadata structures and coverage.

Because content-based recommendation will rely on title, author, subject, and description information, differences in metadata richness between sources could influence similarity calculations.

This section therefore evaluates content depth across:

- Open Library-only books
- LeadershipNow-only books
- books represented in both sources

The purpose is not to remove source differences, but to identify them explicitly before NLP modelling.

If metadata richness is strongly associated with source, later recommendation evaluation should consider whether books are being matched because of genuine semantic similarity or because they share similar levels and types of source metadata.

In [18]:
# ============================================================
# CONTENT DEPTH BY DATA SOURCE
# ============================================================

def classify_source(row):

    if row["source_openlibrary"] and row["source_leadershipnow"]:
        return "Both"

    if row["source_openlibrary"]:
        return "Open Library only"

    if row["source_leadershipnow"]:
        return "LeadershipNow only"

    return "Unknown"


features["source_group"] = features.apply(
    classify_source,
    axis=1
)


# ------------------------------------------------------------
# Source distribution validation
# ------------------------------------------------------------

source_counts = (
    features["source_group"]
    .value_counts()
    .reindex(
        [
            "Open Library only",
            "LeadershipNow only",
            "Both",
            "Unknown"
        ],
        fill_value=0
    )
)


print("SOURCE DISTRIBUTION")
print("=" * 80)
print(source_counts)


# ------------------------------------------------------------
# Content depth by source
# ------------------------------------------------------------

depth_by_source = pd.crosstab(
    features["source_group"],
    features["content_depth"]
)

depth_by_source = depth_by_source.reindex(
    columns=[
        "Minimal",
        "Basic",
        "Enriched",
        "Rich"
    ],
    fill_value=0
)


print("\nCONTENT DEPTH BY SOURCE — COUNTS")
print("=" * 80)

display(depth_by_source)


# ------------------------------------------------------------
# Row percentages
# ------------------------------------------------------------

depth_by_source_pct = (
    depth_by_source
    .div(depth_by_source.sum(axis=1), axis=0)
    * 100
)


print("\nCONTENT DEPTH BY SOURCE — ROW PERCENTAGES")
print("=" * 80)

display(
    depth_by_source_pct.style.format("{:.2f}%")
)


# ------------------------------------------------------------
# Mean document size by source
# ------------------------------------------------------------

source_text_summary = (
    features
    .groupby("source_group")
    .agg(
        books=("book_id", "count"),
        mean_components=("content_component_count", "mean"),
        median_words=("content_word_count", "median"),
        mean_words=("content_word_count", "mean")
    )
    .reset_index()
)


print("\nTEXT DEPTH SUMMARY BY SOURCE")
print("=" * 80)

display(
    source_text_summary.style.format({
        "mean_components": "{:.2f}",
        "median_words": "{:.1f}",
        "mean_words": "{:.1f}"
    })
)

SOURCE DISTRIBUTION
source_group
Open Library only      947
LeadershipNow only    1117
Both                     3
Unknown                  0
Name: count, dtype: int64

CONTENT DEPTH BY SOURCE — COUNTS


content_depth,Minimal,Basic,Enriched,Rich
source_group,,,,
Both,0,0,2,1
LeadershipNow only,0,1117,0,0
Open Library only,3,123,633,188



CONTENT DEPTH BY SOURCE — ROW PERCENTAGES


content_depth,Minimal,Basic,Enriched,Rich
source_group,,,,
Both,0.00%,0.00%,66.67%,33.33%
LeadershipNow only,0.00%,100.00%,0.00%,0.00%
Open Library only,0.32%,12.99%,66.84%,19.85%



TEXT DEPTH SUMMARY BY SOURCE


,source_group,books,mean_components,median_words,mean_words
0,Both,3,3.33,8.0,78.7
1,LeadershipNow only,1117,2.00,6.0,6.3
2,Open Library only,947,3.06,17.0,42.1


## 10. Core and Enriched Content Representations

The source-depth analysis identified a substantial difference in textual metadata coverage between the two sources.

All LeadershipNow-only books contain Basic textual metadata, primarily title and author, whereas most Open Library books additionally contain subjects and/or descriptions.

Using a single enriched document representation could therefore cause recommendation behaviour to be influenced partly by source-specific metadata availability.

To preserve transparency, two content representations are prepared.

### Core Content

`core_content_text` contains:

- title
- authors

This representation provides the most comparable textual basis across the catalogue because author information is available for 99.61% of books and title information is available for all books.

### Enriched Content

`enriched_content_text` contains:

- title
- authors
- subjects
- description

This representation uses all legitimate semantic metadata where available.

No missing subjects or descriptions are fabricated.

The two representations will later allow the NLP and recommendation stages to compare:

- broad catalogue coverage and cross-source comparability using core metadata
- greater semantic richness using enriched metadata

The enriched representation is not automatically assumed to be superior. Its recommendation behaviour will be evaluated empirically.

In [19]:
# ============================================================
# CORE AND ENRICHED CONTENT REPRESENTATIONS
# ============================================================

# ------------------------------------------------------------
# Core representation: title + authors
# ------------------------------------------------------------

features["core_content_text"] = (
    features[
        [
            "title_text",
            "authors_text"
        ]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# ------------------------------------------------------------
# Enriched representation:
# title + authors + subjects + description
# ------------------------------------------------------------

features["enriched_content_text"] = (
    features[
        [
            "title_text",
            "authors_text",
            "subjects_text",
            "description_text"
        ]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# ------------------------------------------------------------
# Document-length features
# ------------------------------------------------------------

features["core_word_count"] = (
    features["core_content_text"]
    .str.split()
    .str.len()
)

features["enriched_word_count"] = (
    features["enriched_content_text"]
    .str.split()
    .str.len()
)


# Additional semantic material beyond core representation
features["additional_semantic_words"] = (
    features["enriched_word_count"]
    - features["core_word_count"]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert features["core_content_text"].ne("").all()
assert features["enriched_content_text"].ne("").all()

assert (
    features["enriched_word_count"]
    >= features["core_word_count"]
).all()

assert (
    features["additional_semantic_words"] >= 0
).all()


representation_summary = (
    features
    .groupby("source_group")
    .agg(
        books=("book_id", "count"),

        median_core_words=(
            "core_word_count",
            "median"
        ),

        mean_core_words=(
            "core_word_count",
            "mean"
        ),

        median_enriched_words=(
            "enriched_word_count",
            "median"
        ),

        mean_enriched_words=(
            "enriched_word_count",
            "mean"
        ),

        median_additional_words=(
            "additional_semantic_words",
            "median"
        ),

        mean_additional_words=(
            "additional_semantic_words",
            "mean"
        )
    )
    .reset_index()
)


print("CORE VS ENRICHED CONTENT REPRESENTATION")
print("=" * 90)

display(
    representation_summary.style.format({
        "median_core_words": "{:.1f}",
        "mean_core_words": "{:.1f}",
        "median_enriched_words": "{:.1f}",
        "mean_enriched_words": "{:.1f}",
        "median_additional_words": "{:.1f}",
        "mean_additional_words": "{:.1f}"
    })
)

CORE VS ENRICHED CONTENT REPRESENTATION


,source_group,books,median_core_words,mean_core_words,median_enriched_words,mean_enriched_words,median_additional_words,mean_additional_words
0,Both,3,5.0,7.0,8.0,78.7,5.0,71.7
1,LeadershipNow only,1117,6.0,6.3,6.0,6.3,0.0,0.0
2,Open Library only,947,7.0,7.4,17.0,42.1,9.0,34.6


In [20]:
# ============================================================
# CORE CONTENT COMPARABILITY CHECK
# ============================================================

core_comparison = (
    features
    .groupby("source_group")
    .agg(
        books=("book_id", "count"),
        core_min_words=("core_word_count", "min"),
        core_median_words=("core_word_count", "median"),
        core_mean_words=("core_word_count", "mean"),
        core_max_words=("core_word_count", "max")
    )
    .reset_index()
)


print("CORE CONTENT COMPARABILITY")
print("=" * 80)

display(
    core_comparison.style.format({
        "core_median_words": "{:.1f}",
        "core_mean_words": "{:.2f}"
    })
)

CORE CONTENT COMPARABILITY


,source_group,books,core_min_words,core_median_words,core_mean_words,core_max_words
0,Both,3,3,5.0,7.00,13
1,LeadershipNow only,1117,3,6.0,6.29,17
2,Open Library only,947,2,7.0,7.42,33


## 11. Final Feature Inventory

Feature engineering produced several complementary feature families while preserving the original canonical book records.

### Metadata Availability Features

These indicate whether specific metadata are observed:

- `has_authors`
- `has_description`
- `has_subjects`
- `has_cover`
- `has_rating`
- `has_engagement`
- `has_extended_text`
- `has_first_publish_year`
- `has_observed_publication_year`
- `has_edition_metadata`
- `has_page_count`

### Temporal Features

- `book_age_from_first_publish`
- `years_since_observed_publication`

These retain the distinction between earliest known work publication and the observed LeadershipNow publication year.

### Transformed Numerical Features

The following skewed count variables received `log1p` transformations while their original values were retained:

- `log1p_ratings_count`
- `log1p_want_to_read_count`
- `log1p_currently_reading_count`
- `log1p_already_read_count`
- `log1p_edition_count`
- `log1p_total_reader_engagement`
- `log1p_page_count`

### Reader-Engagement Features

- `total_reader_engagement`
- `log1p_total_reader_engagement`

The three original reader states remain available independently.

### Edition Features

- observed `format`
- observed `page_count`
- `is_hardcover`
- `is_paperback`
- `log1p_page_count`

These describe the observed LeadershipNow edition where available.

### Text and Content Features

- `title_text`
- `authors_text`
- `subjects_text`
- `description_text`
- `content_text`
- `core_content_text`
- `enriched_content_text`
- `content_word_count`
- `content_char_count`
- `core_word_count`
- `enriched_word_count`
- `additional_semantic_words`

### Content-Richness Features

- `content_component_count`
- `content_depth`

The depth labels describe metadata completeness rather than book quality.

### Source Feature

- `source_group`

This identifies whether the canonical book is represented by Open Library only, LeadershipNow only, or both sources.

### Features Deliberately Not Created

The following were not created because there is currently insufficient methodological justification:

- arbitrary popularity scores
- arbitrary quality scores
- fabricated missing ratings
- fabricated descriptions or subjects
- arbitrary age categories
- one-hot encoding of thousands of authors or subjects
- one-hot encoding of hundreds of publishers
- standardized numerical variables prior to defining a modelling matrix

This preserves interpretability and prevents feature engineering from introducing unsupported assumptions.

In [21]:
# ============================================================
# FINAL FEATURE INVENTORY
# ============================================================

engineered_features = [
    "has_authors",
    "has_description",
    "has_subjects",
    "has_cover",
    "has_rating",
    "has_engagement",
    "has_extended_text",
    "has_first_publish_year",
    "has_observed_publication_year",
    "book_age_from_first_publish",
    "years_since_observed_publication",

    "log1p_ratings_count",
    "log1p_want_to_read_count",
    "log1p_currently_reading_count",
    "log1p_already_read_count",
    "log1p_edition_count",

    "total_reader_engagement",
    "log1p_total_reader_engagement",

    "format",
    "page_count",
    "has_edition_metadata",
    "has_page_count",
    "is_hardcover",
    "is_paperback",
    "log1p_page_count",

    "title_text",
    "authors_text",
    "subjects_text",
    "description_text",
    "content_text",

    "has_title_text",
    "has_author_text",
    "has_subject_text",
    "has_description_text",

    "content_word_count",
    "content_char_count",
    "content_component_count",
    "content_depth",

    "source_group",

    "core_content_text",
    "enriched_content_text",
    "core_word_count",
    "enriched_word_count",
    "additional_semantic_words"
]


missing_engineered = [
    col
    for col in engineered_features
    if col not in features.columns
]


print("FINAL FEATURE INVENTORY")
print("=" * 80)

print(f"Original columns:          {books.shape[1]:,}")
print(f"Current feature columns:   {features.shape[1]:,}")
print(f"Engineered features:       {len(engineered_features):,}")
print(f"Missing expected features: {len(missing_engineered):,}")

if missing_engineered:
    print("\nMissing:")
    print(missing_engineered)
else:
    print("\n✓ ALL EXPECTED ENGINEERED FEATURES PRESENT")

FINAL FEATURE INVENTORY
Original columns:          18
Current feature columns:   69
Engineered features:       44
Missing expected features: 0

✓ ALL EXPECTED ENGINEERED FEATURES PRESENT


In [22]:
# ============================================================
# FINAL FEATURE-ENGINEERING INTEGRITY VALIDATION
# ============================================================

validation_results = {}


# Book grain
validation_results["rows_equal_2067"] = (
    len(features) == 2067
)

validation_results["unique_book_ids_equal_2067"] = (
    features["book_id"].nunique() == 2067
)

validation_results["no_duplicate_book_ids"] = (
    features["book_id"].duplicated().sum() == 0
)


# Text integrity
validation_results["all_books_have_core_text"] = (
    features["core_content_text"].ne("").all()
)

validation_results["all_books_have_enriched_text"] = (
    features["enriched_content_text"].ne("").all()
)

validation_results["enriched_not_shorter_than_core"] = (
    features["enriched_word_count"]
    .ge(features["core_word_count"])
    .all()
)


# Content richness
validation_results["content_components_valid"] = (
    features["content_component_count"]
    .between(1, 4)
    .all()
)


# Edition metadata
validation_results["edition_metadata_count_1120"] = (
    features["has_edition_metadata"].sum() == 1120
)

validation_results["page_count_coverage_1119"] = (
    features["has_page_count"].sum() == 1119
)


# Source composition
validation_results["openlibrary_only_947"] = (
    (features["source_group"] == "Open Library only").sum()
    == 947
)

validation_results["leadershipnow_only_1117"] = (
    (features["source_group"] == "LeadershipNow only").sum()
    == 1117
)

validation_results["both_sources_3"] = (
    (features["source_group"] == "Both").sum()
    == 3
)

validation_results["no_unknown_source"] = (
    (features["source_group"] == "Unknown").sum()
    == 0
)


validation_table = (
    pd.Series(validation_results)
    .rename_axis("validation")
    .reset_index(name="passed")
)


print("FINAL FEATURE-ENGINEERING VALIDATION")
print("=" * 80)

display(validation_table)


assert validation_table["passed"].all()

print("\n✓ ALL FEATURE-ENGINEERING VALIDATION CHECKS PASSED")

FINAL FEATURE-ENGINEERING VALIDATION


,validation,passed
0,rows_equal_2067,True
1,unique_book_ids_equal_2067,True
2,no_duplicate_book_ids,True
3,all_books_have_core_text,True
4,all_books_have_enriched_text,True
5,enriched_not_shorter_than_core,True
6,content_components_valid,True
7,edition_metadata_count_1120,True
8,page_count_coverage_1119,True
9,openlibrary_only_947,True



✓ ALL FEATURE-ENGINEERING VALIDATION CHECKS PASSED


In [23]:
# ============================================================
# SAVE FEATURE-ENGINEERED DATASET
# ============================================================

features_output_path = (
    DATA_PROCESSED
    / "books_features.csv"
)

features.to_csv(
    features_output_path,
    index=False
)


# ------------------------------------------------------------
# Reload from disk for reproducibility validation
# ------------------------------------------------------------

features_check = pd.read_csv(
    features_output_path,
    low_memory=False
)


assert features_check.shape == features.shape
assert len(features_check) == 2067
assert features_check["book_id"].nunique() == 2067
assert features_check["book_id"].duplicated().sum() == 0


print("FEATURE DATASET SAVED")
print("=" * 80)

print(f"File:     {features_output_path}")
print(f"Rows:     {features_check.shape[0]:,}")
print(f"Columns:  {features_check.shape[1]:,}")
print(
    f"Book IDs: {features_check['book_id'].nunique():,}"
)

print("\n✓ SAVED FILE SUCCESSFULLY RELOADED AND VALIDATED")

FEATURE DATASET SAVED
File:     /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/books_features.csv
Rows:     2,067
Columns:  69
Book IDs: 2,067

✓ SAVED FILE SUCCESSFULLY RELOADED AND VALIDATED


In [24]:
# ============================================================
# SAVE FEATURE-ENGINEERING SUMMARY
# ============================================================

feature_summary = pd.DataFrame({
    "metric": [
        "canonical_books",
        "feature_columns",
        "engineered_features",
        "books_with_authors",
        "books_with_subjects",
        "books_with_descriptions",
        "books_with_ratings",
        "books_with_engagement",
        "books_with_edition_metadata",
        "books_with_page_count",
        "minimal_content_books",
        "basic_content_books",
        "enriched_content_books",
        "rich_content_books",
        "openlibrary_only_books",
        "leadershipnow_only_books",
        "both_source_books"
    ],

    "value": [
        len(features),
        features.shape[1],
        len(engineered_features),

        features["has_authors"].sum(),
        features["has_subjects"].sum(),
        features["has_description"].sum(),
        features["has_rating"].sum(),
        features["has_engagement"].sum(),

        features["has_edition_metadata"].sum(),
        features["has_page_count"].sum(),

        (features["content_depth"] == "Minimal").sum(),
        (features["content_depth"] == "Basic").sum(),
        (features["content_depth"] == "Enriched").sum(),
        (features["content_depth"] == "Rich").sum(),

        (features["source_group"] == "Open Library only").sum(),
        (features["source_group"] == "LeadershipNow only").sum(),
        (features["source_group"] == "Both").sum()
    ]
})


summary_output_path = (
    DATA_PROCESSED
    / "feature_engineering_summary.csv"
)

feature_summary.to_csv(
    summary_output_path,
    index=False
)


print("FEATURE-ENGINEERING SUMMARY")
print("=" * 80)

display(feature_summary)

print(f"\nSaved to:\n{summary_output_path}")

FEATURE-ENGINEERING SUMMARY


,metric,value
0,canonical_books,2067
1,feature_columns,69
2,engineered_features,44
3,books_with_authors,2059
4,books_with_subjects,826
5,books_with_descriptions,192
6,books_with_ratings,282
7,books_with_engagement,807
8,books_with_edition_metadata,1120
9,books_with_page_count,1119



Saved to:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/feature_engineering_summary.csv


# Feature Engineering Conclusions

Feature engineering transformed the integrated catalogue into a modelling-ready dataset while preserving the canonical grain of **2,067 unique books**.

The final engineered dataset contains **69 columns**, including the original integrated metadata and **44 explicitly defined engineered features**.

## Key Engineering Outcomes

### 1. Metadata availability was preserved explicitly

Missing information was not replaced with fabricated values. Instead, availability indicators were created for authors, subjects, descriptions, ratings, engagement measures, publication information, edition metadata, and page count.

This allows later modelling stages to distinguish between a genuine observed value and unavailable metadata.

### 2. Skewed numerical variables were transformed without removing their originals

Reader-engagement counts, rating counts, edition counts, and page counts showed substantial positive skew.

`log1p` transformations were therefore created while retaining the original variables for interpretation and validation.

### 3. Temporal variables were engineered reproducibly

Book age and observed-publication recency were calculated using a fixed reference year of **2026**.

First publication year and observed publication year remain separate because they represent different bibliographic concepts.

### 4. Edition metadata was integrated without changing the canonical book grain

LeadershipNow edition information was available for **1,120 books**, including page count for **1,119 books**.

The merge preserved all **2,067 canonical books** without duplication.

### 5. High-cardinality categorical variables were not indiscriminately encoded

The dataset contains **2,289 raw author values**, **2,300 subject labels**, and **302 publishers**.

Authors and subjects are therefore retained primarily for semantic/NLP representation, while publisher remains bibliographic metadata rather than being expanded into hundreds of dummy variables.

Edition format contains only two observed categories and was represented using binary indicators.

### 6. Textual metadata richness differs substantially by source

All **1,117 LeadershipNow-only books** contain Basic textual metadata, while most Open Library records contain additional subjects and/or descriptions.

This creates a source-related difference in semantic information availability.

To address this transparently, two content representations were prepared:

- **Core content:** title + author
- **Enriched content:** title + author + available subjects + available description

The core representation is substantially more comparable across sources, with median document lengths of **6 words for LeadershipNow-only books** and **7 words for Open Library-only books**.

The enriched representation preserves additional legitimate semantic information but will be evaluated separately because its metadata depth is source-dependent.

### 7. Content depth was measured rather than assumed

The catalogue contains:

- **3 Minimal** records
- **1,240 Basic** records
- **635 Enriched** records
- **189 Rich** records

These classifications describe metadata completeness only and do not represent book quality.

## Modelling Implications

The engineered dataset is ready for subsequent NLP, dimensionality-reduction, clustering, and recommendation-system development.

Feature engineering deliberately does **not** yet perform:

- TF-IDF vectorization
- stop-word removal
- stemming or lemmatization
- numerical standardization
- PCA
- clustering
- similarity computation
- missing-value fabrication
- arbitrary popularity or quality scoring

These decisions belong to the relevant downstream modelling stages so that each transformation remains methodologically explicit and reproducible.

The canonical `books_master.csv` remains the authoritative integrated dataset. The engineered dataset is stored separately as `books_features.csv`.